In [2]:
import redback
print(redback.__version__)

import numpy as np
import matplotlib.pyplot as plt
from redback.model_library import all_models_dict

from astropy.cosmology import Planck18 as cosmo 

import redback.interaction_processes as ip
import redback.sed as sed
import redback.photosphere as photosphere

import astropy.units as uu

import extinction
from extinction import ccm89, fitzpatrick99, apply, remove
from scipy.interpolate import RegularGridInterpolator
import sncosmo


import inspect
from redback.transient_models import supernova_models

#import lambda to nu
from redback.transient_models.supernova_models import lambda_to_nu

No module named 'lalsimulation'
lalsimulation is not installed. Some EOS based models will not work. Please use bilby eos or pass your own EOS generation class to the model
14:51 bilby INFO    : Running bilby version: 2.3.0
14:51 redback INFO    : Running redback version: 1.12.1


1.12.1


In [4]:
#define model fro redback 

def sn1998bw_template(time, redshift, amplitude, **kwargs):
    """
    A wrapper to the SN1998bw template. Only valid between 1100-11000 Angstrom and 0.01 to 90 days post explosion in rest frame

    Parameters
    ----------
    time
        time in days in observer frame (post explosion)
    redshift
        redshift
    amplitude
        amplitude scaling factor, where 1.0 is the original brightness of SN1998bw; and f_lambda is scaled by this factor
    kwargs
        Additional keyword arguments required by redback.
    frequency
        Required if output_format is 'flux_density'. frequency to calculate - Must be same length as time array or a single number).
    bands
        Required if output_format is 'magnitude' or 'flux'.
    output_format
        'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'
    cosmology
        Cosmology to use for luminosity distance calculation. Defaults to Planck18. Must be a astropy.cosmology object.
    Returns
    -------
        set by output format - 'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'

    """
    import sncosmo
    model = sncosmo.Model(source='v19-1998bw')
    original_redshift = 0.0085
    cosmology = kwargs.get("cosmology", cosmo)
    original_dl = (43*uu.Mpc).to(uu.cm).value

    # From roughly matching to Galama+ or Clocchiatti+1998bw light curves
    original_peak_time = 15
    model.set(z=original_redshift, t0=original_peak_time)
    model.set_source_peakmag(14.25, band='bessellb', magsys='ab')
    tts = np.geomspace(0.01, 90, 200)
    lls = np.linspace(1620, 11000, 300)
    f_lambda = model.flux(tts, lls) #erg/s/cm^2/Angstrom.
    l_lambda = f_lambda * 4 * np.pi * original_dl**2  # erg/s/Angstrom

    # We consider this the rest frame spectrum of 1998bw. Now we can redshift it and scale it.
    time_obs = tts * (1 + redshift)
    lambda_obs = lls * (1 + redshift)
    dl_new = cosmology.luminosity_distance(redshift).cgs.value
    f_lambda_obs = l_lambda / (4 * np.pi * dl_new**2)
    f_lambda_obs = amplitude * f_lambda_obs * (1 + redshift) # accounting for bandwidth stretching
    f_lambda_obs = f_lambda_obs * uu.erg / uu.s / uu.cm ** 2 / uu.Angstrom
    if kwargs['output_format'] == 'flux_density':
        frequency = kwargs['frequency']
        # work in obs frame
        ff_array = lambda_to_nu(lambda_obs)

        # Convert flux density to mJy
        fmjy = f_lambda_obs.to(uu.mJy, equivalencies=uu.spectral_density(wav=lambda_obs * uu.Angstrom)).value
        # Create interpolator on obs frame grid
        flux_interpolator = RegularGridInterpolator(
            (time_obs, ff_array),
            fmjy,
            bounds_error=False,
            fill_value=0.0)

        # Prepare points for interpolation
        if isinstance(frequency, (int, float)):
            frequency = np.ones_like(time) * frequency

        # Create points for evaluation
        points = np.column_stack((time, frequency))

        # Return interpolated flux density with (1+z) correction for observer frame
        return flux_interpolator(points)
    elif kwargs['output_format'] == 'spectra':
        return namedtuple('output', ['time', 'lambdas', 'spectra'])(time=time_obs,
                                                                    lambdas=lambda_obs,
                                                                    spectra=f_lambda_obs)
    else:
        return sed.get_correct_output_format_from_spectra(time=time, time_eval=time_obs,
                                                          spectra=f_lambda_obs, lambda_array=lambda_obs,
                                                          **kwargs)



In [5]:
#specific GRB event info 

#GRB 060218 has mag_AB = 17.22, epoch = 11.0, z_event = 0.0331 in R-band, f_98 = 0.7 -- WORKS ? 
# event_name = 'GRB 060218'
# time = np.array((11))
# red = 0.0331
# freq = 	4.61700e+14 #r-band 
# AB_mag = 17.22
# a_v= 0.39

#TEST ANOTHER GRB: GRB 211211A f_98 <0.01
#new data: i band 3.74100e+14 Hz , AB_mag = 20.92 (already corrected for galactic extinction  A_v = 0.048 ?) , 0.68 days , z = 0.0763

# event_name = 'GRB 211211A'
# AB_mag = 20.92
# red = 0.0763
# time = np.array(0.68)  # Days in observer frame
# #band = 'bessellr' not required? -- not if we are using frequency 
# freq = 3.74100e+14 # I-band 
# #a_v = 0.018 * 3.1 
#a_v = 0.048 #or 0.0 

#GRB 060614
# event_name = 'GRB 060614'
# AB_mag = 22.8
# red = 0.125
# time = np.array(9.5)  # Days in observer frame
# #band = 'bessellr' not required? -- not if we are using frequency 
# freq = 8.36900e+14 # U-band  
# a_v = 0.02* 3.1

event_name = '171205A'
AB_mag = 17.7
red = 0.0368
time = np.array(10.98)
freq = 3.74100e+14
# band = 'besselli'
a_v = 0.05 * 3.1 #E(B-V) x R_v


#extinction along the line-of-sight of E(B−V) = 0.05 mag nor the one intrinsic to the host with a value of E(B−V)int = 0.02
#get wavelength from filter tables redback 
#ensure i am always matching my wavelength to my frequency band if i am changing it !! 
wavelength =  np.array([8020.14000]) #angstroms

#interpolate sn1998bw model 
sn_1998bw = sn1998bw_template(time=time,
redshift=red,
amplitude=1.0,
frequency=freq,
output_format='flux_density')

print("Flux density of 1998bw =", sn_1998bw[0],"mJy")


Flux density of 1998bw = 0.32477056648312685 mJy


In [6]:
#specify the filter (conversion if needed) 
#dustmaps does the correct magnitude correction !! 


#convert AB magnitude of arbitrary GRB event --> flux density (mJy)
def calc_flux_density_from_ABmag(AB_mag):
    """
    Calculate flux density from AB magnitude assuming monochromatic AB filter

    :param magnitudes: AB magnitude values
    :return: flux density
    """
    return (AB_mag * uu.ABmag).to(uu.mJy)

flux_density_event = calc_flux_density_from_ABmag(AB_mag)

print("This is the flux density of  GRB event", event_name ,f"without extinction correction: {flux_density_event:.5f}")

dereddened_flux_event = extinction.remove(fitzpatrick99(wavelength, a_v, 3.1),flux_density_event)

#dereddened flux is now a 1-d array so display the first element 
print(f"This is the extinction corrected GRB event flux density found using fitzpatrick99: {dereddened_flux_event[0]:.5f}")

#remove host flux density from GRB event by following the AB mag to flux density and then remove from the de-reddened original flux 
#hopefully get smaller ratio overall 

AB_mag_host = 19.59 # 
#band = I 

def calc_flux_density_from_ABmag(AB_mag_host):
    """
    Calculate flux density from AB magnitude assuming monochromatic AB filter

    :param magnitudes: AB magnitude values
    :return: flux density
    """
    return (AB_mag_host * uu.ABmag).to(uu.mJy)

flux_density_host = calc_flux_density_from_ABmag(AB_mag_host)

print("This is the flux density of GRB host", f": {flux_density_host:.5f}")

#now subtract this from the de-reddened to get the host-corrected flux density

host_corrected_flux_density = dereddened_flux_event - flux_density_host

print(host_corrected_flux_density)

#print("The flux density corrected for host = ", f": {host_corrected_flux_density:.5f}")

#or do i need to subtract one magnitude from the other and then convert to flux density ? 

AB_mag_final = AB_mag - AB_mag_host

#now convert to a flux density ? 
#WHAT IS GOING ON ??? -- GATHER INFO FIRST , GET CODE WORKING LATER 

#now use this AB_mag final to get a flux and then compare to SN 1998bw 

This is the flux density of  GRB event 171205A without extinction correction: 0.30200 mJy
This is the extinction corrected GRB event flux density found using fitzpatrick99: 0.32671 mJy
This is the flux density of GRB host : 0.05297 mJy
[0.27374356] mJy


In [12]:
#extinction affects host and GRB event equally ?? or no 
# 1. Convert AB mags to flux densities

AB_mag_host = 19.59
AB_mag_event = 17.7
flux_event = calc_flux_density_from_ABmag(AB_mag_event)
flux_host = calc_flux_density_from_ABmag(AB_mag_host)

# 2. Subtract host from event
host_corrected_flux = flux_event - flux_host

# 3. Apply extinction correction
dereddened_flux = extinction.remove(fitzpatrick99(wavelength, a_v, 3.1), host_corrected_flux)

print(dereddened_flux)

#same thing !! 

#if host is brighter than event they may have alrready subtracted host from it !! - this explains the negative value 

#14.8 = WRONG and its more likely 21 something 

[0.26940889] mJy


In [14]:
#now do final calculation of f_98 ratio: 

f_98 = dereddened_flux_event[0] /sn_1998bw[0]

print(f"The final flux density ratio result of F_GRB / F_1998bw = {f_98:.3f}")

#why so large ?? becvause F_GRB_event >>> F_SN1998bw 
#try with actual values to see if things change 
#ensure frquency range stays small !! 

#this value is more close to true value !! 
#try below method / see what it actually does 

#overcalculating the dereddened flux value - getting top heavy ratio - even more inaccurate than the last one 

The final flux density ratio result of F_GRB / F_1998bw = 1.006 mJy


In [9]:
#is this a fluke or is this actually the correct calculation with an offset applied ? 

#test other values and see what comes out !! 
#the test value i ran stated it should've been less than 0.01 but i got 0.5 ?? 
#issue with very low / high redshifts

In [10]:
#do i have to do something like this instead ? run through the model twice?? 
#what does this do ? not exactly what i'm after? 
'''
sn_1998b2 = func(time=time,redshift=red,amplitude=1.0,frequency=freq,output_format='flux_density')
sn_1998b1 = func(time=time,redshift=0.09,amplitude=1.0,frequency=freq,output_format='flux_density')

print(sn_1998b2, sn_1998b1, sn_1998b2/sn_1998b1)'''

"\nsn_1998b2 = func(time=time,redshift=red,amplitude=1.0,frequency=freq,output_format='flux_density')\nsn_1998b1 = func(time=time,redshift=0.09,amplitude=1.0,frequency=freq,output_format='flux_density')\n\nprint(sn_1998b2, sn_1998b1, sn_1998b2/sn_1998b1)"

In [11]:
'''redshift = 0.0085
times = np.linspace(0.1,90,200)
sn_1998bw = func(time=times,redshift=redshift,amplitude=1.0,output_format='flux_density',frequency=freq)
sn_1998bw1 = func(time=times,redshift=0.09,amplitude=1.0,frequency=freq,output_format='flux_density')

plt.semilogy(times,sn_1998bw,'-b')
plt.semilogy(times,sn_1998bw1, '-r')#
plt.semilogy(time,sn_1998b1, marker='o',color='green')
#plt.ylim(19,13)
plt.show()'''

NameError: name 'func' is not defined

In [ ]:
#SN peaks around 20 days getting later time obs = better around 20 days !! 
#also need the host sub -- have to subtract the host galaxy if very bright - subtract observed mag of host 

#check what is published and known, ive made a comparison against an upper limit / a detection without any afterglow or host subtraction 
